# Reproduce the final predictions from the weights in the repo

This notebook runs the whole pipeline from the model weights, without retraining, and checks that it is the pipeline of the report:

1. finds the weights (`weights/` of the repo, verified against `weights/MANIFEST.json`) and checks their SHA-256,
2. re-scores the date holdout (142 files, 231 annotator views) with the existing pooled PQ code and compares it with the documented **0.4255**,
3. shows 6 overlays of the predicted masks on holdout images,
4. predicts the 180 test images, writes `submission.csv` and validates it with `filament.validate`,
5. prints and saves the seconds per image, for inference alone and for the whole pipeline.

**The pipeline is fixed.** U-Net with a ResNet-34 encoder trained 80 epochs on native-resolution crops (tensor SHA-256 `3ac77e98…`, `docs/RESULTS.md`); tiled inference (tile from the checkpoint, overlap 128) with flip test-time augmentation (four views); pixel threshold 0.45; closing with `close_size` 11; solar-disk gate; components smaller than `min_area` 400 dropped; object filter: a component is kept if its mean probability is at least 0.8. The detector of phase F is not part of it (not adopted).

None of the cells asks for input. Import the notebook into Kaggle and press **Save & Run All**, or run it locally (`docs/reproduce.md`). It does not submit anything: there is no Kaggle submit call, and the CSV is only written to the output folder.

**`PILOT = True`** runs a few frames only (`PILOT_FRAMES` holdout frames and test images), to check the plumbing. A pilot is not a result: its PQ is not compared with 0.4255 and its timing is not a measurement.

**Attach exactly these Kaggle inputs (jury route, nothing private)**

1. The competition `filament-segmentation-2026`.
2. **This public repository, uploaded by you as your own Kaggle dataset** (Datasets → New Dataset → upload the repository folder or its GitHub zip). The notebook finds `src` at any depth; the version has to contain `filament/reproduce.py`.
3. The weights. `weights/unet_r34_crops_fp16.pt` is committed, so it is in that dataset and nothing else is attached. Their tensor hash is verified and a wrong one stops the run. The fallback, the output of the author's notebook `train-unet-r34-crops-80ep`, is the author's own route.

*Author's own route:* the private dataset `ignacioaguirrecandia/solar-code` instead of step 2. A jury account cannot attach it.

**Settings:** Accelerator **GPU T4** (without a GPU the notebook stops, unless `ALLOW_CPU`); Internet **On**, only for `pip install segmentation-models-pytorch==0.6.0` (PyTorch is already on Kaggle).

In [ ]:
# ---- Settings (the only cell to edit) ----------------------------------------
# A few frames only, to check the plumbing. Not a result.
PILOT = False
PILOT_FRAMES = 4

# Efficiency is measured on a GPU: without one the notebook stops. True runs it
# on the CPU anyway (very slow; use it with PILOT).
ALLOW_CPU = False

# Folder with train/ and test/. None: FILAMENT_DATA_ROOT, then the Kaggle path,
# then data/MAGFiLO_1.0_Kaggle_2026 and data/ under the repository root (not the
# working directory, which is notebooks/ in Jupyter).
DATA_ROOT = None
# Where submission.csv, overlays_holdout.png and timing.json go. None: /kaggle/working
# on Kaggle, outputs/reproduce/ of the repo otherwise (both are not committed).
OUTPUT_DIR = None

# The weights: this file in weights/ of the repo, else (stopgap, Kaggle only) a file
# of this name from the attached training output.
WEIGHTS_FILE = "unet_r34_crops_fp16.pt"
KAGGLE_WEIGHTS_FILE = "unet_r34_crops.pt"

# The final pipeline. Fixed: a test keeps these equal to the documented values.
THRESHOLD = 0.45
CLOSE_SIZE = 11
MIN_AREA = 400
OBJECT_FILTER = {"stat": "mean", "min": 0.8}
USE_TTA = True  # flip TTA: as is, left-right, top-bottom, both
TILE_OVERLAP = 128
INFER_BATCH_SIZE = 8
USE_AMP = True

# The holdout PQ check: documented value and tolerance. Outside it, the run stops
# (except in a PILOT).
EXPECTED_HOLDOUT_PQ = 0.4255
PQ_TOLERANCE = 0.002

# Overlays on holdout images: how many, and the seed that picks them.
OVERLAY_COUNT = 6
OVERLAY_SEED = 0
SEED = 0

## 1. `src` on the path

`src` is the directory that contains `filament/reproduce.py`. A checkout in the working directory (or any parent) is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match, for example `/kaggle/input/datasets/<your-user>/<your-dataset>/Solar-Filament-Segmentation-main/src`. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the reproduce module (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "reproduce.py").is_file():
            return src
    return None


def _attached_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest attached-dataset ``src`` under ``root``.

    ``filament/reproduce.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "reproduce.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    attached dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _attached_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/reproduce.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). On Kaggle, upload this repository as a dataset of "
            "your own (Datasets > New Dataset) and attach it, for example "
            "/kaggle/input/datasets/<your-user>/<your-dataset>/"
            "Solar-Filament-Segmentation-main/src. If a dataset is attached, "
            "its version predates this notebook (it has no "
            "filament/reproduce.py): upload the current version of the "
            "repository."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Libraries and the GPU

`segmentation-models-pytorch` **0.6.0** is installed only if it is missing and the notebook is on Kaggle (locally it comes from `requirements.txt`). PyTorch is not installed here: Kaggle has its own, and locally it comes from `requirements.txt`. The runs behind the documented numbers used torch 2.11.0; another version prints a warning and the PQ check below says whether it matters.

Without a GPU the notebook stops here (unless `ALLOW_CPU`).

In [ ]:
import importlib.util
import random
import subprocess

import numpy as np

SMP_VERSION = "0.6.0"
TORCH_VERSION = "2.11.0"
ON_KAGGLE = Path("/kaggle/working").is_dir()

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is not installed. Locally install the pinned environment: "
        "`uv sync --extra gpu` or `pip install -r requirements.txt` (docs/reproduce.md)."
    )
if importlib.util.find_spec("segmentation_models_pytorch") is None:
    if not ON_KAGGLE:
        raise RuntimeError(
            "segmentation-models-pytorch is not installed. Locally: "
            "`uv sync --extra gpu` or `pip install -r requirements.txt`."
        )
    completed = subprocess.run(
        [PYTHON, "-m", "pip", "install", f"segmentation-models-pytorch=={SMP_VERSION}"],
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
            "Internet has to be On."
        )

import segmentation_models_pytorch as smp
import torch

from filament import crop_training as ct
from filament.reproduce import FINAL_CONFIG, PipelineConfig, check_accelerator

print(f"segmentation-models-pytorch {smp.__version__}, torch {torch.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, found {smp.__version__}."
    )
if not torch.__version__.startswith(TORCH_VERSION):
    print(
        f"WARNING: torch {torch.__version__}, the documented runs used {TORCH_VERSION}. "
        "Small numeric differences are possible; the PQ check below decides."
    )

DEVICE = check_accelerator(torch.cuda.is_available(), ALLOW_CPU)
device = torch.device(DEVICE)
if DEVICE == "cuda":
    print(f"device={torch.cuda.get_device_name(0)}")
else:
    print(
        "WARNING: device=cpu (ALLOW_CPU). Very slow, and the timing is not a GPU timing."
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PIPELINE = PipelineConfig(
    threshold=THRESHOLD,
    close_size=CLOSE_SIZE,
    min_area=MIN_AREA,
    object_filter=OBJECT_FILTER,
    tta=USE_TTA,
)
if PIPELINE != FINAL_CONFIG:
    print(f"WARNING: these settings are not the documented pipeline {FINAL_CONFIG}.")
print(f"pipeline={PIPELINE}")

## 3. Find the weights, verify them, build the network

`filament.weights.locate_weights` looks for `weights/<WEIGHTS_FILE>` in the repo and, only if it is not there, for `<KAGGLE_WEIGHTS_FILE>` under `/kaggle/input` (a folder that Kaggle extracted from the `.pt` zip is re-zipped to the output folder). The notebook prints which one it used.

* **From the repo** (fp16, `weights/MANIFEST.json`): the file SHA-256, the tensor SHA-256 of the fp16 state, the stored epochs and encoder must match the manifest, and the manifest must descend from the pinned fp32 hash of the 80-epoch run (`filament.checkpoint.R34_CROPS`, `3ac77e98…`).
* **From the Kaggle training output** (fp32): `filament.checkpoint.verify_and_report` against `R34_CROPS` (epochs, encoder, tensor SHA-256).

A wrong hash stops the run. fp16 weights are loaded into the fp32 network (PyTorch converts them on `load_state_dict`). fp16 changes the values slightly: the holdout PQ check in section 5 is what shows the conversion is harmless.

In [ ]:
from filament.checkpoint import R34_CROPS, checkpoint_fingerprint, verify_and_report
from filament.ensemble import EvalConfig, ensemble_predictor
from filament.weights import (
    WEIGHTS_DIR_NAME,
    locate_weights,
    manifest_entry,
    materialize_checkpoint,
    verify_repo_weights,
)

REPO_ROOT = SRC.parent
OUTPUT = (
    Path(OUTPUT_DIR)
    if OUTPUT_DIR
    else (Path("/kaggle/working") if ON_KAGGLE else REPO_ROOT / "outputs" / "reproduce")
)
OUTPUT.mkdir(parents=True, exist_ok=True)

source = locate_weights(REPO_ROOT, WEIGHTS_FILE, KAGGLE_WEIGHTS_FILE)
print(f"weights from the {source.kind}: {source.path}")
weights_file = materialize_checkpoint(source.path, OUTPUT)
checkpoint = ct.load_checkpoint(weights_file)
if source.kind == "repo":
    entry = manifest_entry(REPO_ROOT / WEIGHTS_DIR_NAME, WEIGHTS_FILE)
    fingerprint = verify_repo_weights(checkpoint, weights_file, entry)
    print(
        "repo weights verified: file sha256, fp16 tensor sha256 and descent from the "
        f"pinned fp32 hash {R34_CROPS.tensor_sha256[:8]}…"
    )
else:
    fingerprint = checkpoint_fingerprint(checkpoint)
    verify_and_report(
        fingerprint,
        expected=R34_CROPS,
        remedy=(
            "This notebook cannot run unverified weights. Attach the output of "
            "the training notebook train-unet-r34-crops-80ep."
        ),
    )
print(
    f"epochs={fingerprint['epochs']} encoder={fingerprint['encoder']} "
    f"tensor_sha256={fingerprint['tensor_sha256']}"
)

TILE_SIZE = int(checkpoint.get("tile_size") or checkpoint["crop_size"])
model = ct.model_from_checkpoint(checkpoint, device)
del checkpoint
use_amp = USE_AMP and device.type == "cuda"
tiles = ct.tiled_predictor(
    model,
    device,
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
    batch_size=INFER_BATCH_SIZE,
    use_amp=use_amp,
)
predict = ensemble_predictor(
    {"r34": tiles}, EvalConfig("final", {"r34": 1.0}, tta=USE_TTA)
)
print(
    f"tile_size={TILE_SIZE} overlap={TILE_OVERLAP} batch={INFER_BATCH_SIZE} amp={use_amp}"
)

## 4. The data

`DATA_ROOT` is the competition folder that holds `train/` and `test/`. The holdout is the same date split as the training notebooks: the most recent 20% of calendar dates, one row per file (`filament.train_data.split_training_images`). A `PILOT` keeps `PILOT_FRAMES` holdout frames, spread over the holdout, and the first `PILOT_FRAMES` test images.

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.reproduce import (
    EXPECTED_HOLDOUT,
    KAGGLE_DATA_ROOT,
    resolve_data_root,
)
from filament.sweep import count_views
from filament.train_data import split_training_images

DATA = resolve_data_root(DATA_ROOT, repo_root=REPO_ROOT)
IMAGE_DIR = DATA / Path(TRAIN_IMAGES_DIR).relative_to(KAGGLE_DATA_ROOT)
ANNOTATIONS_PATH = DATA / Path(TRAIN_ANNOTATIONS_PATH).relative_to(KAGGLE_DATA_ROOT)
TEST_IMAGES = DATA / "test" / "test_images"
print(f"data={DATA}")

loaded = load_train_annotations(ANNOTATIONS_PATH)
holdout = split_training_images(loaded.images).holdout
print(f"holdout files={len(holdout)} annotator views={count_views(holdout)}")
if not PILOT and (len(holdout), count_views(holdout)) != EXPECTED_HOLDOUT:
    print(
        f"WARNING: the documented holdout is {EXPECTED_HOLDOUT[0]} files with "
        f"{EXPECTED_HOLDOUT[1]} views. [VERIFY] the data version."
    )
if PILOT:
    step = max(1, len(holdout) // PILOT_FRAMES)
    holdout = holdout[::step][:PILOT_FRAMES]
    print(f"PILOT: {len(holdout)} holdout frames")

test_paths = sorted(
    p for p in TEST_IMAGES.iterdir() if p.is_file() and p.suffix.lower() == ".jpeg"
)
if not test_paths:
    raise FileNotFoundError(f"No .jpeg files in {TEST_IMAGES}")
if not PILOT and len(test_paths) != 180:
    print(f"WARNING: {len(test_paths)} test images, the documented test set has 180.")
if PILOT:
    test_paths = test_paths[:PILOT_FRAMES]
    print(f"PILOT: {len(test_paths)} test images")
print(f"test images={len(test_paths)} directory={TEST_IMAGES}")

## 5. Re-score the holdout and compare with the documented PQ

`filament.reproduce.score_holdout` is `filament.components.per_file_object_counts` and `filament.stats.weighted_pq`, the code that produced the 0.4255 of `docs/RESULTS.md`, fed with probability maps predicted now (not cached ones). The number is the pooled per-view PQ, with the checkpoint, the threshold, the closing and the area cut chosen on this same holdout: it is optimistic, and the cross-fitted estimate in `docs/RESULTS.md` is lower.

The check allows ±0.002 around 0.4255 and **stops the run outside it**, before any CSV is written. In a `PILOT` it only prints.

In [ ]:
from filament.reproduce import (
    Timing,
    check_holdout_pq,
    pick_overlay_names,
    score_holdout,
)

overlay_names = pick_overlay_names(
    [image.file_name for image in holdout], OVERLAY_COUNT, OVERLAY_SEED
)


def _holdout_progress(index, total, image):
    if index == 1 or index % 20 == 0 or index == total:
        print(f"  holdout {index}/{total} {image.file_name}", flush=True)


holdout_timing = Timing()
holdout_result = score_holdout(
    holdout,
    IMAGE_DIR,
    predict,
    PIPELINE,
    holdout_timing,
    keep=overlay_names,
    progress=_holdout_progress,
)
print(
    f"holdout files={holdout_result.files} views={holdout_result.views} "
    f"pooled per-view PQ={holdout_result.pq:.4f}"
)
print(
    check_holdout_pq(
        holdout_result.pq,
        expected=EXPECTED_HOLDOUT_PQ,
        tolerance=PQ_TOLERANCE,
        pilot=PILOT,
    )
)

## 6. Six overlays on holdout images

Frames picked with a fixed seed (`OVERLAY_SEED`) from the holdout; each instance has its own colour and outline. The figure is also saved as `overlays_holdout.png` in the output folder.

In [ ]:
import math

import cv2
import matplotlib.pyplot as plt

from filament.crop_data import read_gray
from filament.reproduce import labels_for_frame, overlay

names = sorted(holdout_result.probabilities)
columns = 3
rows_needed = max(1, math.ceil(len(names) / columns))
figure, axes = plt.subplots(
    rows_needed, columns, figsize=(6 * columns, 6 * rows_needed), squeeze=False
)
for axis in axes.flat:
    axis.axis("off")
for axis, name in zip(axes.flat, names):
    gray = read_gray(IMAGE_DIR / name)
    labels = labels_for_frame(gray, holdout_result.probabilities[name], PIPELINE)
    picture = overlay(gray, labels)
    axis.imshow(cv2.resize(picture, (768, 768), interpolation=cv2.INTER_AREA))
    axis.set_title(f"{name}: {int(labels.max())} filaments", fontsize=9)
figure.tight_layout()
figure.savefig(OUTPUT / "overlays_holdout.png", dpi=100)
plt.show()

## 7. Predict the test images and write `submission.csv`

Test images are read at 2048 × 2048 and nothing else of the test set is used. Each frame goes through the same `filament.reproduce.labels_for_frame` as the holdout. `filament.submission.write_submission` writes `submission.csv` (columns `filament_id`, `segmentation_rle`) and `filament.validate.validate_submission` checks it. The file is not submitted.

In [ ]:
from filament.reproduce import iter_submission_rows
from filament.submission import write_submission
from filament.validate import validate_submission

SUBMISSION_PATH = OUTPUT / "submission.csv"


def _test_progress(index, total, path, count):
    if index == 1 or index % 20 == 0 or index == total:
        print(f"  test {index}/{total} {path.name} filaments={count}", flush=True)


test_timing = Timing()
write_submission(
    SUBMISSION_PATH,
    iter_submission_rows(
        test_paths, predict, PIPELINE, test_timing, progress=_test_progress
    ),
)
validate_submission(SUBMISSION_PATH)
written = sum(1 for _ in SUBMISSION_PATH.open(encoding="utf-8")) - 1
print(f"wrote {written} rows for {test_timing.images} images to {SUBMISSION_PATH}")
print("columns=filament_id,segmentation_rle; validated with filament.validate")

## 8. Time per image

Two numbers per image: **inference** (`predict`: tiled passes, four flips, averaging) and the **whole pipeline** (reading the JPEG, inference, labelling, object filter, RLE, CSV). The test set is the measurement; the holdout loop, which also scores the masks, is shown for reference. A `PILOT` or a CPU run is not a measurement. They are saved in `timing.json`.

In [ ]:
from filament.reproduce import write_timing

measurement = "PILOT (not a measurement)" if PILOT else "test set"
if DEVICE != "cuda":
    measurement += ", CPU (not a GPU measurement)"
summaries = [
    test_timing.summary(f"test, {measurement}"),
    holdout_timing.summary("holdout, includes scoring"),
]
for summary in summaries:
    print(
        f"{summary['label']}: {summary['images']} images, "
        f"inference {summary['inference_seconds_per_image']} s/image, "
        f"whole pipeline {summary['total_seconds_per_image']} s/image"
    )
write_timing(OUTPUT / "timing.json", summaries)
print(f"device={torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'}")
print(f"wrote {OUTPUT / 'timing.json'}")
print("This notebook wrote the CSV. It did not submit it.")